# <u>Intro a Series de Tiempo con FbProphet</u>


Prophet es un conocido modelo local de series temporales estructurales bayesianas.

## Cómo funciona Prophet

Prophet es muy útil para conjuntos de datos:

* Que contengan un periodo de tiempo extendido (meses o años) de observaciones históricas detalladas (por hora, día o semana)

* Que tengan varias estacionalidades muy marcadas

* Que incluyan eventos anteriormente conocidos importantes, pero irregulares

* Que les falten puntos de datos o tengan casos atípicos grandes

* Que tengan tendencias de crecimiento no lineal que se aproximen a un límite.

Prophet es un modelo de regresión aditiva con una tendencia de curva de crecimiento lineal o logística por partes. Incluye un componente estacional anual modelado usando series de Fourier y un componente estacional semanal modelado usando variables ficticias.

# Importando librerías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from math import sqrt
import seaborn as sns

# Importando Data
- Dataset: Pasajeros de aerolínea
- Unidad: Miles

In [ ]:
#Importamos la data
url = 'https://raw.githubusercontent.com/JBrianAlicorp/Business-Analytics/master/international-airline-passengers.csv'
df = pd.read_csv(url,encoding='latin1',header = None)

In [ ]:
df.columns = ['year','passengers']

In [ ]:
df.head(10)

# Preprocesamiento de data y visualización

__Convertimos el formato de fecha:__

In [ ]:
df['year'] = pd.to_datetime(df['year'], format='%Y-%m')

__Establecer índice como la columna de fecha y hora para manipulaciones más fáciles:__

---



In [ ]:
y = df.set_index('year')

In [ ]:
y.plot(figsize=(15, 6))
plt.show()

__Cajas y bigotes:__
- Los valores medianos a través de los años confirman una tendencia al alza
- Aumento constante de la propagación, o 50% medio de los datos (cuadros) con el tiempo
- Un modelo que considere la estacionalidad podría funcionar bien

In [ ]:
fig, ax = plt.subplots(figsize=(15,6))
sns.boxplot(x=y.passengers.index.year, y=y.passengers, ax=ax)
plt.show()

## Prophet
- [Prophet] (https://facebook.github.io/prophet/) es un software de código abierto lanzado por el equipo de Core Data Science de Facebook.
- Prophet es un procedimiento para pronosticar datos de series de tiempo basado en un modelo aditivo / multiplicativo donde las tendencias no lineales se ajustan a la estacionalidad anual, semanal y diaria, más los efectos de vacaciones.
- Funciona mejor con series de tiempo que tienen fuertes efectos estacionales y varias temporadas de datos históricos.
- Prophet es robusto ante los datos faltantes y los cambios en la tendencia, y generalmente maneja bien los valores atípicos.
- El paquete Prophet proporciona parámetros intuitivos que son fáciles de ajustar.

__Parámetros de tendencia__

- crecimiento: 'lineal' o 'logístico' para especificar una tendencia lineal o logística
- puntos de cambio: lista de fechas en las que se incluyen posibles puntos de cambio (automático si no se especifica)
- n_changepoints: si no se proporcionan puntos de cambio, puede proporcionar el número de puntos de cambio que se incluirán automáticamente
- changepoint_prior_scale: parámetro para cambiar la flexibilidad de la selección automática de puntos de cambio


__Estacionalidad y parámetros de vacaciones__

- Estacionalidad anual: ajusta la estacionalidad anual
- semanal_estacionalidad: Ajustar estacionalidad semanal
- daily_seasonality: ajusta la estacionalidad diaria
- vacaciones: marco de datos del feed que contiene el nombre y la fecha de vacaciones
- seasonality_prior_scale: parámetro para cambiar la fuerza del modelo de estacionalidad
- holiday_prior_scale: parámetro para cambiar la fuerza del modelo de vacaciones

Prophet requiere que los nombres de las variables en la serie de tiempo sean:

- y - Target
- ds - Fecha y hora

In [ ]:
#divide into train and validation set
train = y[:int(0.75*(len(y)))]
valid = y[int(0.75*(len(y))):]

#plotting the data
train['passengers'].plot()
valid['passengers'].plot()
plt.show()

In [ ]:
train.head()

In [ ]:
print(train.shape)
print(valid.shape)

In [ ]:
train_prophet = pd.DataFrame()
train_prophet['ds'] = train.index
train_prophet['y'] = train.passengers.values

In [ ]:
train_prophet.head()

In [ ]:
!pip install prophet

In [ ]:
# import os
# # Let cmdstanpy know where CmdStan is
# os.environ["CMDSTAN"] = "./cmdstan-2.23.0"

from prophet import Prophet

In [ ]:
#instantiate Prophet with only yearly seasonality as our data is monthly
model = Prophet()
model.fit(train_prophet) #fit the model with your dataframe

In [ ]:
# predict for five months in the furure and MS - month start is the frequency
future = model.make_future_dataframe(periods = 36, freq = 'MS')
future

In [ ]:
# now lets make the forecasts
forecast = model.predict(future)
forecast[['ds', 'yhat']]

In [ ]:
fig = model.plot(forecast)
#plot the predictions for validation set

plt.plot(valid, label='Valid', color = 'red', linewidth = 2)

plt.show()

In [ ]:
model.plot_components(forecast)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, median_absolute_error, mean_squared_log_error

In [ ]:
def evaluate_forecast(y,pred):
    results = pd.DataFrame({'r2_score':r2_score(y, pred),
                           }, index=[0])
    results['mean_absolute_error'] = mean_absolute_error(y, pred)
    results['median_absolute_error'] = median_absolute_error(y, pred)
    results['mse'] = mean_squared_error(y, pred)
    results['msle'] = mean_squared_log_error(y, pred)
    results['rmse'] = np.sqrt(results['mse'])
    return results

In [ ]:
evaluate_forecast(y, forecast.yhat) # total

In [ ]:
evaluate_forecast(valid, forecast.yhat[-36:]) # valid

In [ ]:
#definiendo que la estacionalidad sea anual y el modelo multiplicativo
model = Prophet(yearly_seasonality=True,seasonality_mode= 'multiplicative')
model.fit(train_prophet) #fit the model with your dataframe

In [ ]:
# predict for five months in the furure and MS - month start is the frequency
future = model.make_future_dataframe(periods = 36, freq = 'MS')
forecast = model.predict(future)
fig = model.plot(forecast)
#plot the predictions for validation set

plt.plot(valid, label='Valid', color = 'red', linewidth = 2)

plt.show()

In [ ]:
evaluate_forecast(y, forecast.yhat)

In [ ]:
evaluate_forecast(valid, forecast.yhat[-36:]) # valid

## Otro Ejemplo usando Prophet

In [ ]:
# Descarga de datos, DEMANDA ELECTRICA (MW) desde 2011-12-31 al 2014-12-31
# ==============================================================================
url = 'https://raw.githubusercontent.com/skforecast/skforecast-datasets/main/data/h2o.csv'
datos = pd.read_csv(url, sep=',')

In [ ]:
datos['fecha'] = pd.to_datetime(datos['fecha'], format='%Y-%m-%d')
print(datos)
datos.dtypes

In [ ]:
datos = datos.set_index('fecha') # fecha como nombre de fila
datos = datos.rename(columns={'x': 'y'})

In [ ]:
datos

In [ ]:
#divide into train and validation set
train = datos[:int(0.75*(len(datos)))]
valid = datos[int(0.75*(len(datos))):]


#plotting the data
train['y'].plot()
valid['y'].plot()
plt.show()

In [ ]:
train_prophet = pd.DataFrame()
train_prophet['ds'] = train.index
train_prophet['y'] = train.y.values

In [ ]:
#instantiate Prophet with only yearly seasonality as our data is monthly
model = Prophet( yearly_seasonality=True, seasonality_mode = 'multiplicative')
model.fit(train_prophet) #fit the model with your dataframe

In [ ]:
valid.shape

In [ ]:
# predict for five months in the furure and MS - month start is the frequency
future = model.make_future_dataframe(periods = 51, freq = 'MS')
future

In [ ]:
# now lets make the forecasts
forecast = model.predict(future)
forecast[['ds', 'yhat']]

In [ ]:
fig = model.plot(forecast)
#plot the predictions for validation set

plt.plot(valid, label='Valid', color = 'red', linewidth = 2)

plt.show()

In [ ]:
evaluate_forecast(datos, forecast.yhat)

In [ ]:
evaluate_forecast(valid, forecast.yhat[-51:]) # valid

## probando con el parámetro growth logistic

In [ ]:
train_prophet = pd.DataFrame()
train_prophet['ds'] = train.index
train_prophet['y'] = train.y.values
train_prophet['cap'] = 1.2

In [ ]:
#instantiate Prophet with only yearly seasonality as our data is monthly
model = Prophet(yearly_seasonality=True,weekly_seasonality=True,daily_seasonality=True,
                seasonality_mode = 'multiplicative',
                growth = 'logistic')
model.fit(train_prophet) #fit the model with your dataframe

In [ ]:
future = model.make_future_dataframe(periods = 51, freq = 'MS')
future['cap'] = 1.2
forecast = model.predict(future)
fig = model.plot(forecast)
plt.plot(valid, label='Valid', color = 'red', linewidth = 2)
plt.show()

In [ ]:
evaluate_forecast(datos.y, forecast.yhat)

In [ ]:
evaluate_forecast(valid.y, forecast.yhat[-51:]) # valid

# Haciendo predicciones en el futuro

In [ ]:
future = model.make_future_dataframe(periods = 63, freq = 'MS')
future['cap'] = 1.2
forecast = model.predict(future)
fig = model.plot(forecast)
plt.plot(valid, label='Valid', color = 'red', linewidth = 2)
plt.show()

In [ ]:
forecast[['ds','yhat']]